In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
import random
%matplotlib inline 

In [2]:
path = Path("names.txt")

words = path.read_text().splitlines()

print(len(words))
print(words[:10])

32033
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [3]:
chs = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chs)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)

In [4]:
bs = 3

def b_ds(words):
    x, y = [], []
    
    for w in words:
    
        context = [0] * bs
        for ch in w + '.':
            ix = stoi[ch]
            x.append(context)
            y.append(ix)
            context = context[1:] + [ix]
    
    x = torch.tensor(x)
    y = torch.tensor(y)
    return x, y

In [8]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad). item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print (f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [11]:
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

In [12]:
xtrain, ytrain = b_ds(words[:n1])
xval, yval = b_ds(words[n1:n2])
xtest, ytest = b_ds(words[n2:])

In [10]:
# Selected config: {'emb_dim': 24, 'hidden_dim': 750, 'batch_size': 256, 'lr': 0.15355887634905266}
#from the best cfg i found in my last file
n_e = 24
n_h = 750

C = torch.randn((vocab_size, n_e))
w1 = torch.randn((n_e * bs, n_h)) * (5/3) / ((n_e * bs)**.5)
b1 = torch.randn(n_h) * 0.01
w2 = torch.randn((n_h, vocab_size)) * 0.01
b2 = torch.randn(vocab_size) * 0.1

bngain = torch.ones((1, n_h)) * 0.1 + 1.0
bnbias = torch.zeros((1, n_h)) * 0.1
bnmean_running = torch.zeros((1, n_h))
bnstd_running = torch.ones((1, n_h))

parameters = [C, w1, b1, w2, b2, bngain, bnbias]

for p in parameters:
    p.requires_grad = True

In [15]:
max_steps = 50000
batch_size = 256
n = batch_size
lossi = []
lr = 0.15355887634905266

In [14]:
ix = torch.randint(0, xtrain.shape[0], (batch_size,))
xb, yb = xtrain[ix], ytrain[ix]

In [17]:
emb = C[xb]
embcat = emb.view(emb.shape[0], -1)

hprebn = embcat @ w1 + b1
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

h = torch.tanh(hpreact)

logits = h @ w2 + b2

logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), yb].mean()

for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
         norm_logits, logit_maxes, logits, h, hpreact, bnraw, 
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.2740, grad_fn=<NegBackward0>)

In [21]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), yb] = -1.0/n
cmp('logprobs', dlogprobs, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


In [31]:
dprobs = dlogprobs * probs**-1
cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


In [35]:
dcounts_sum_inv = (dprobs * counts).sum(1,keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0


In [39]:
dcounts_sum = dcounts_sum_inv * -counts_sum**-2
cmp('counts_sum', dcounts_sum, counts_sum)

counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0


In [41]:
dcounts = dprobs * counts_sum_inv + dcounts_sum * torch.ones_like(counts)
cmp('counts', dcounts, counts)

counts          | exact: True  | approximate: True  | maxdiff: 0.0


In [42]:
dnorm_logits = dcounts * counts
cmp('norm_logits', dnorm_logits, norm_logits)

norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0


In [52]:
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)

logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0


In [58]:
dlogits = dnorm_logits.clone()

ix = logits.max(dim=1).indices

dlogits[torch.arange(logits.shape[0]), ix] += dlogit_maxes.squeeze(1)

cmp('logits', dlogits, logits)

logits          | exact: True  | approximate: True  | maxdiff: 0.0


In [66]:
dh = dlogits @ w2.T
cmp('h', dh, h)

h               | exact: True  | approximate: True  | maxdiff: 0.0


In [69]:
dw2 = h.T @ dlogits
cmp('dw2', dw2, w2)

dw2             | exact: True  | approximate: True  | maxdiff: 0.0


In [72]:
db2 = dlogits.sum(dim=0)
cmp('db2', db2, b2)

db2             | exact: True  | approximate: True  | maxdiff: 0.0


In [78]:
dhpreact = dh * (1-h**2)
cmp('hpreact', dhpreact, hpreact)

hpreact         | exact: False | approximate: True  | maxdiff: 1.4551915228366852e-11


In [85]:
dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
cmp('bngain', dbngain, bngain)

bngain          | exact: False | approximate: True  | maxdiff: 1.4551915228366852e-10


In [88]:
dbnbias = (dhpreact).sum(0, keepdim=True)
cmp('bnbias', dbnbias, bnbias)

bnbias          | exact: False | approximate: True  | maxdiff: 6.984919309616089e-10


In [92]:
dbnraw = dhpreact * bngain
cmp('bnraw', dbnraw, bnraw)

bnraw           | exact: False | approximate: True  | maxdiff: 1.4551915228366852e-11


In [97]:
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

bnvar_inv       | exact: False | approximate: True  | maxdiff: 2.3283064365386963e-10


In [101]:
dbnvar = dbnvar_inv * (-0.5*(bnvar+1e-5)**-1.5)
cmp('bnvar', dbnvar, bnvar)

bnvar           | exact: False | approximate: True  | maxdiff: 5.820766091346741e-11


In [108]:
dbndiff2 = (dbnvar * torch.ones_like(bndiff2)) / (n-1)
cmp('bndiff2', dbndiff2, bndiff2)

bndiff2         | exact: False | approximate: True  | maxdiff: 2.2737367544323206e-13


In [114]:
dbndiff = (dbndiff2 * 2 * bndiff) + (dbnraw * bnvar_inv)
cmp('bndiff', dbndiff, bndiff)

bndiff          | exact: False | approximate: True  | maxdiff: 1.0913936421275139e-11


In [117]:
dbnmeani = -dbndiff.sum(0, keepdim=True) 
cmp('bnmeani', dbnmeani, bnmeani)

bnmeani         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10


In [124]:
dhprebn = ((dbnmeani * torch.ones_like(hprebn)) / n) + (dbndiff)
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 1.0913936421275139e-11


In [128]:
dembcat = dhprebn @ w1.T
cmp('embcat', dembcat, embcat)

embcat          | exact: False | approximate: True  | maxdiff: 7.275957614183426e-11


In [131]:
dw1 = embcat.T @ dhprebn
cmp('w1', dw1, w1)

w1              | exact: False | approximate: True  | maxdiff: 6.984919309616089e-10


In [134]:
db1 = dhprebn.sum(0, keepdim=True)
cmp('b1', db1, b1)

b1              | exact: False | approximate: True  | maxdiff: 3.5652192309498787e-10


In [139]:
demb = dembcat.view(emb.shape)
cmp('emb', demb, emb)

emb             | exact: False | approximate: True  | maxdiff: 7.275957614183426e-11


In [144]:
dC = torch.zeros_like(C)

for i in range(xb.shape[0]):
    for j in range(xb.shape[1]):
        idx = xb[i, j]
        dC[idx] += demb[i, j]

cmp('C', dC, C)

C               | exact: False | approximate: True  | maxdiff: 1.3969838619232178e-09


In [146]:
dlogits_f = F.softmax(logits, dim=1)
dlogits_f[range(n), yb] -= 1
dlogits_f /= n

cmp('logits', dlogits_f, logits)

logits          | exact: False | approximate: True  | maxdiff: 9.89530235528946e-10


In [147]:
dhprebn_f = bngain * bnvar_inv * (
    dhpreact
    - dhpreact.mean(0, keepdim=True)
    - bndiff / ((n-1) * (bnvar + 1e-5))
      * (dhpreact * bndiff).sum(0, keepdim=True)
)

cmp('hprebn', dhprebn_f, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 1.4551915228366852e-11


In [ ]:
#